# Week 7 — DPLL and the Wumpus World

**Lesson plan:** [`../weeks/week-07.md`](../weeks/week-07.md) · **Slides:** [`../slides/week-07/deck.md`](../slides/week-07/deck.md)

Live coding, 25 minutes. Build DPLL (1962), then use it to implement the central
identity of the unit:

# `KB ⊨ α`  iff  `KB ∧ ¬α` is unsatisfiable

**Checkpoint 1 is this week** (weeks 1–6).

## 1 · Representation

CNF as a list of `frozenset`s of ints. Positive `n` means variable *n*; negative
means its negation. This is the DIMACS convention and every SAT tool uses it.

In [ ]:
import itertools
from collections import Counter

# We name variables with ints and keep a symbol table for printing.
class Vars:
    def __init__(self):
        self.by_name, self.by_id = {}, {}
    def __call__(self, name):
        if name not in self.by_name:
            i = len(self.by_name) + 1
            self.by_name[name] = i
            self.by_id[i] = name
        return self.by_name[name]
    def name(self, lit):
        return ("¬" if lit < 0 else "") + self.by_id[abs(lit)]


def clause_str(c, V):
    return "(" + " ∨ ".join(V.name(l) for l in sorted(c, key=abs)) + ")"


V = Vars()
P, Q, R = V("P"), V("Q"), V("R")
# (P -> Q) in CNF is (-P v Q)
example = [frozenset({-P, Q}), frozenset({P})]
print(" ∧ ".join(clause_str(c, V) for c in example))

## 2 · DPLL

Backtracking search over assignments, with **unit propagation** and **pure literal
elimination**.

> Unit propagation is week 6's forward checking, specialized to Boolean domains. A
> clause with one unassigned literal leaves **no choice** — so assign it. Notice
> that this is structurally the same solver you wrote last week.

In [ ]:
def dpll(clauses, assignment=None, stats=None):
    """Returns a satisfying assignment dict, or None if unsatisfiable."""
    assignment = dict(assignment or {})
    if stats is None:
        stats = Counter()
    stats["calls"] += 1

    # ---- 1. unit propagation, to fixpoint -------------------------------
    changed = True
    while changed:
        changed = False
        for clause in clauses:
            unassigned, satisfied = [], False
            for l in clause:
                v = assignment.get(abs(l))
                if v is None:
                    unassigned.append(l)
                elif v == (l > 0):
                    satisfied = True
                    break
            if satisfied:
                continue
            if not unassigned:
                stats["conflicts"] += 1
                return None                       # clause falsified
            if len(unassigned) == 1:              # forced
                lit = unassigned[0]
                assignment[abs(lit)] = (lit > 0)
                stats["propagations"] += 1
                changed = True

    # ---- 2. pure literal elimination ------------------------------------
    remaining = [c for c in clauses
                 if not any(assignment.get(abs(l)) == (l > 0) for l in c)]
    all_lits = {l for c in remaining for l in c if abs(l) not in assignment}
    for lit in all_lits:
        if -lit not in all_lits and abs(lit) not in assignment:
            assignment[abs(lit)] = (lit > 0)
            stats["pure"] += 1

    # ---- 3. base case ----------------------------------------------------
    if all(any(assignment.get(abs(l)) == (l > 0) for l in c) for c in clauses):
        return assignment

    # ---- 4. branch -------------------------------------------------------
    unassigned = [abs(l) for c in clauses for l in c if abs(l) not in assignment]
    if not unassigned:
        return None
    var = unassigned[0]
    for value in (True, False):
        stats["decisions"] += 1
        result = dpll(clauses, {**assignment, var: value}, stats)
        if result is not None:
            return result
    return None


s = Counter()
model = dpll(example, stats=s)
print("model:", {V.by_id[k]: v for k, v in sorted(model.items())})
print("stats:", dict(s))

## 3 · The central identity, in code

Three lines. This is the whole reason a SAT solver is a general-purpose reasoning
engine rather than a puzzle tool.

In [ ]:
def negate_cnf(clauses):
    """Negate a CNF formula. Only correct for a CONJUNCTION OF UNIT clauses,
    which is all we need for entailment queries about single literals."""
    lits = [next(iter(c)) for c in clauses if len(c) == 1]
    assert len(lits) == len(clauses), "negate_cnf: need unit clauses only"
    # ¬(a ∧ b) = (¬a ∨ ¬b)  -- one clause
    return [frozenset(-l for l in lits)]


def entails(kb_clauses, alpha_clauses):
    """KB ⊨ α  iff  KB ∧ ¬α is unsatisfiable."""
    return dpll(kb_clauses + negate_cnf(alpha_clauses)) is None


# Modus ponens: {P -> Q, P} entails Q
kb = [frozenset({-P, Q}), frozenset({P})]
print("KB ⊨ Q      :", entails(kb, [frozenset({Q})]))       # True
print("KB ⊨ ¬Q     :", entails(kb, [frozenset({-Q})]))      # False

# Affirming the consequent: {P -> Q, Q} does NOT entail P
kb2 = [frozenset({-P, Q}), frozenset({Q})]
print("\n{P→Q, Q} ⊨ P:", entails(kb2, [frozenset({P})]),
      "  <-- the warm-up poll. Invalid, and the solver agrees.")
assert entails(kb, [frozenset({Q})])
assert not entails(kb2, [frozenset({P})])

## 4 · The Wumpus World

A 4×4 grid. Pits cause a breeze in adjacent squares. The agent must move only to
squares it can **prove** are safe.

Encoding: `P(x,y)` means "pit at (x,y)"; `B(x,y)` means "breeze at (x,y)".
The physics: a square is breezy **iff** some neighbour has a pit.

In [ ]:
N = 4
W = Vars()

def Pv(x, y):
    return W(f"P{x}{y}")

def Bv(x, y):
    return W(f"B{x}{y}")

def neighbours(x, y):
    return [(x+dx, y+dy) for dx, dy in ((1,0), (-1,0), (0,1), (0,-1))
            if 1 <= x+dx <= N and 1 <= y+dy <= N]


def wumpus_physics():
    """B(x,y) <-> (P(n1) v P(n2) v ...) for every square, in CNF."""
    clauses = []
    for x in range(1, N+1):
        for y in range(1, N+1):
            b = Bv(x, y)
            ns = [Pv(a, c) for a, c in neighbours(x, y)]
            # B -> (P1 v P2 v ...)   ==>  (-B v P1 v P2 v ...)
            clauses.append(frozenset([-b] + ns))
            # (P1 v P2 v ...) -> B   ==>  (-Pi v B) for each i
            for p in ns:
                clauses.append(frozenset({-p, b}))
    return clauses


# Percepts: (1,1) has no pit and no breeze; (2,1) has a breeze.
kb = wumpus_physics()
kb += [frozenset({-Pv(1,1)}),      # we are standing here, so no pit
       frozenset({-Bv(1,1)}),      # no breeze at (1,1)
       frozenset({-Pv(2,1)}),      # we walked here safely
       frozenset({Bv(2,1)})]       # breeze at (2,1)

print(f"KB: {len(kb)} clauses over {len(W.by_name)} variables")

## 5 · The agent reasons

For each frontier square, ask three questions. **The third answer is the important
one.**

In [ ]:
def classify(kb, x, y):
    safe    = entails(kb, [frozenset({-Pv(x, y)})])   # provably no pit
    deadly  = entails(kb, [frozenset({Pv(x, y)})])    # provably a pit
    if safe:
        return "SAFE (proved)"
    if deadly:
        return "PIT (proved)"
    return "UNKNOWN"


print("Percepts: (1,1) no breeze, (2,1) breeze.\n")
for (x, y) in [(1,2), (2,2), (3,1), (1,3)]:
    print(f"  ({x},{y}): {classify(kb, x, y)}")

print("""
(1,2) is SAFE -- no breeze at (1,1) proves no pit in any neighbour of (1,1).
(2,2) and (3,1) are UNKNOWN. The breeze at (2,1) proves a pit exists in one of
them, but NOT which one.

THAT IS THE POINT. A sound agent that cannot prove safety does not guess. It
reports UNKNOWN. An agent that guesses here has a 50% chance of dying, and
"50% correct" is not a description anyone wants of their reasoning system.""")

## 6 · Add a percept and watch UNKNOWN become PROVED

Move to (1,2), perceive no breeze. Now the KB is strong enough to resolve (2,2).

In [ ]:
kb2 = kb + [frozenset({-Bv(1,2)})]      # no breeze at (1,2)

print("After adding 'no breeze at (1,2)':\n")
for (x, y) in [(2,2), (3,1), (1,3)]:
    before = classify(kb, x, y)
    after = classify(kb2, x, y)
    mark = "  <-- CHANGED" if before != after else ""
    print(f"  ({x},{y}): {before:<14} -> {after}{mark}")

print("""
(2,2) is now provably safe, and therefore -- by the breeze at (2,1) -- (3,1) is
provably A PIT. One extra percept resolved two squares.

This is monotonic reasoning: adding true facts never invalidates a previous
conclusion. Week 12 is where that guarantee has to be given up.""")

## 7 · ⚠️ Inference beats search — the same lesson as week 6

Count how much work unit propagation does versus branching.

In [ ]:
s = Counter()
dpll(kb2 + negate_cnf([frozenset({-Pv(2,2)})]), stats=s)
total = s["propagations"] + s["decisions"]
print("On the Wumpus entailment query:")
for k in ("calls", "propagations", "pure", "decisions", "conflicts"):
    print(f"  {k:<14}{s[k]:>8,}")
if total:
    print(f"\n  forced by propagation : {s['propagations']/total:>6.1%}")
    print(f"  actual guesses        : {s['decisions']/total:>6.1%}")

print("""
Most variables are assigned by PROPAGATION, with very little branching.
Same lesson as week 6, in a different formalism: INFERENCE BEATS SEARCH.

That is also why modern SAT solvers scale to millions of variables -- they are
mostly propagation engines with a little search bolted on, not the reverse.""")

## 8 · The honest boundary

Try to prove something the KB genuinely does not determine.

In [ ]:
print("Is (4,4) safe?  ", classify(kb2, 4, 4))
print("""
UNKNOWN -- we have no percepts anywhere near it.

Logic tells you WHAT FOLLOWS. It does not tell you what to do when nothing
follows. A logical agent facing only UNKNOWN squares is stuck, and no amount of
better inference fixes that -- the information is not in the KB.

That gap is the motivation for week 12: reasoning under uncertainty, where you
act on degrees of belief rather than proofs.""")

## 9 · Studio

A Wumpus agent that moves **only** to squares it can prove safe. Then the same
percepts, in natural language, given to an LLM.

**Score on soundness, not accuracy.** The three outcomes:

| Outcome | Meaning |
|---|---|
| correct + proved | the good case |
| **says UNKNOWN when it is UNKNOWN** | ✅ **also the good case** |
| confidently asserts an unprovable claim | ❌ **unsound — the failure that matters** |

An LLM that says "(2,2) is probably safe" when safety is *not entailed* has made
the one error your DPLL agent structurally cannot make. Record every instance in
the Failure Atlas.